# Fabric Notebook AI Auditor

Uses Microsoft Fabric AI functions (`ai.generate_response`) to assess notebook
definition files for external resource access and data exfiltration risk.

## How It Works
1. Reads notebook definition files from a Lakehouse Files directory
2. Extracts code content from each notebook (`.ipynb`, `.py`, `.txt`, `.json`)
3. Splits long notebooks into chunks that fit AI function input limits
4. Passes each chunk to the AI function with a security-focused prompt
5. Recombines chunk-level scores, sources, destinations, and rationale per notebook
6. Writes notebook-level results to a Delta table for downstream analysis

## Prerequisites
- AI functions must be enabled in your Fabric workspace
- A Lakehouse with notebook exports in the Files area
- `synapse.ml.spark.aifunc` available in the Spark environment

In [ ]:
# === Configuration ===========================================================

# Path under the attached Lakehouse's Files area containing notebook exports.
SOURCE_PATH = "Files/notebooks"

# File glob patterns to match.
SOURCE_FILE_GLOB = ["*.ipynb", "*.py", "*.txt"]

# Output table name (written to the default attached Lakehouse).
OUTPUT_TABLE = "notebook_ai_audit_results"

# Max files to process (None = all, set integer for testing)
MAX_FILES = None

# Maximum characters of notebook content to send to each AI function call.
# Long notebooks are split into chunks of this size and recombined after scoring.
MAX_CONTENT_CHARS = 14000

## Load Notebook Files from Lakehouse

In [ ]:
from pyspark.sql import functions as F
from functools import reduce

_globs = [SOURCE_FILE_GLOB] if isinstance(SOURCE_FILE_GLOB, str) else list(SOURCE_FILE_GLOB)

def _load_glob(pattern):
    return (spark.read.format("binaryFile")
        .option("recursiveFileLookup", "true")
        .option("pathGlobFilter", pattern)
        .load(SOURCE_PATH))

_dfs = [_load_glob(g) for g in _globs]
file_df = reduce(lambda a, b: a.union(b), _dfs).dropDuplicates(["path"])

n_total = file_df.count()
if n_total == 0:
    raise SystemExit(f"No files matched patterns {_globs} under '{SOURCE_PATH}'.")

if MAX_FILES:
    file_df = file_df.limit(MAX_FILES)
    n_total = min(n_total, MAX_FILES)

print(f"Patterns: {_globs}")
print(f"Found {n_total} notebook file(s) to assess.")
for row in file_df.select("path").limit(5).collect():
    print(f"  {row.path}")

## Extract and Chunk Notebook Content

Convert binary file content into notebook code, then split each notebook into AI-sized chunks without truncating.

In [ ]:
import json
import base64
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType


@udf(returnType=StringType())
def extract_notebook_content(content, path):
    """Extract code content from a notebook file as a single text block."""
    try:
        text = bytes(content).decode("utf-8", errors="ignore")
    except Exception:
        return None

    fname = path.rsplit("/", 1)[-1] if "/" in path else path

    # .py files: return as-is
    if fname.endswith(".py"):
        return text

    # Try JSON-based formats (.ipynb, .json, .txt)
    try:
        data = json.loads(text)
    except (json.JSONDecodeError, ValueError):
        # Non-JSON .txt: return raw text
        return text if fname.endswith(".txt") else None

    if not isinstance(data, dict):
        return text if fname.endswith(".txt") else None

    code_blocks = []

    # Standard .ipynb format
    if "cells" in data:
        for cell in data.get("cells", []):
            if cell.get("cell_type") == "code":
                src = cell.get("source", [])
                code_blocks.append("".join(src) if isinstance(src, list) else str(src))

    # Fabric Item JSON with definition.parts
    if not code_blocks:
        parts = (data.get("definition") or {}).get("parts") or []
        for part in parts:
            p = part.get("path", "")
            if p.endswith(".py") or p.endswith(".ipynb"):
                try:
                    payload = base64.b64decode(part.get("payload", "")).decode("utf-8", errors="ignore")
                    if p.endswith(".ipynb"):
                        nb = json.loads(payload)
                        for cell in nb.get("cells", []):
                            if cell.get("cell_type") == "code":
                                src = cell.get("source", [])
                                code_blocks.append("".join(src) if isinstance(src, list) else str(src))
                    else:
                        code_blocks.append(payload)
                except Exception:
                    continue

    # Fabric Item JSON with properties.cells
    if not code_blocks:
        cells = (data.get("properties") or {}).get("cells") or []
        for cell in cells:
            if cell.get("cell_type") == "code":
                src = cell.get("source", [])
                code_blocks.append("".join(src) if isinstance(src, list) else str(src))

    if code_blocks:
        return "\n\n# --- cell boundary ---\n\n".join(code_blocks)

    # Fallback for .txt
    return text if fname.endswith(".txt") else None


# Apply extraction and filter out empty results
content_df = (
    file_df
    .withColumn("notebook_content", extract_notebook_content(F.col("content"), F.col("path")))
    .filter(F.col("notebook_content").isNotNull())
    .filter(F.length(F.col("notebook_content")) > 0)
    .select(
        F.col("path"),
        F.element_at(F.split(F.col("path"), "/"), -1).alias("file_name"),
        F.col("notebook_content"),
        F.length(F.col("notebook_content")).alias("content_length"),
    )
)

content_df.cache()

# Split each notebook into AI-sized chunks instead of truncating long notebooks.
notebook_chunks_df = (
    content_df
    .withColumn("chunk_count", F.ceil(F.col("content_length") / F.lit(MAX_CONTENT_CHARS)).cast("int"))
    .withColumn("chunk_index", F.explode(F.sequence(F.lit(0), F.col("chunk_count") - F.lit(1))))
    .withColumn("chunk_number", F.col("chunk_index") + F.lit(1))
    .withColumn(
        "notebook_definition",
        F.expr(f"substring(notebook_content, chunk_index * {MAX_CONTENT_CHARS} + 1, {MAX_CONTENT_CHARS})")
    )
    .select(
        "path",
        "file_name",
        "content_length",
        "chunk_index",
        "chunk_number",
        "chunk_count",
        "notebook_definition",
    )
)

notebook_chunks_df.cache()

n_valid = content_df.count()
n_chunks = notebook_chunks_df.count()
print(f"{n_valid} notebook(s) with extractable code content (of {n_total} total files).")
print(f"Created {n_chunks} chunk(s) using a maximum of {MAX_CONTENT_CHARS} characters each.")
print("\nContent length stats:")
content_df.select("content_length").summary("min", "mean", "max").show()
print("\nChunk count distribution:")
notebook_chunks_df.groupBy("chunk_count").count().orderBy("chunk_count").show(truncate=False)

## Define AI Audit Prompt and Response Schema

In [ ]:
AI_AUDIT_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "fabric_notebook_exfiltration_audit",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "external_resource_access_score": {
                    "type": "integer",
                    "minimum": 0,
                    "maximum": 100,
                    "description": "0=no external access, 100=confirmed external access",
                },
                "exfiltration_risk_score": {
                    "type": "integer",
                    "minimum": 0,
                    "maximum": 100,
                    "description": "0=no exfiltration risk, 100=clear high-risk data movement",
                },
                "sources": {
                    "type": "array",
                    "description": "External or workspace data sources read by the notebook chunk",
                    "items": {
                        "type": "object",
                        "properties": {
                            "endpoint": {
                                "type": "string",
                                "description": "Static endpoint or pseudocode for dynamic endpoint construction",
                            },
                            "type": {
                                "type": "string",
                                "description": "Protocol or service type such as https, abfss, s3, jdbc, smtp, webhook",
                            },
                            "deterministic": {
                                "type": "boolean",
                                "description": "True only when the full endpoint is a literal value in the notebook chunk",
                            },
                        },
                        "required": ["endpoint", "type", "deterministic"],
                        "additionalProperties": False,
                    },
                },
                "destinations": {
                    "type": "array",
                    "description": "External or workspace destinations written to by the notebook chunk",
                    "items": {
                        "type": "object",
                        "properties": {
                            "endpoint": {
                                "type": "string",
                                "description": "Static endpoint or pseudocode for dynamic endpoint construction",
                            },
                            "type": {
                                "type": "string",
                                "description": "Protocol or service type such as https, abfss, s3, jdbc, smtp, webhook",
                            },
                            "deterministic": {
                                "type": "boolean",
                                "description": "True only when the full endpoint is a literal value in the notebook chunk",
                            },
                        },
                        "required": ["endpoint", "type", "deterministic"],
                        "additionalProperties": False,
                    },
                },
                "rationale": {
                    "type": "string",
                    "description": "Brief justification for the scores and most important evidence in this chunk",
                },
            },
            "required": [
                "external_resource_access_score",
                "exfiltration_risk_score",
                "sources",
                "destinations",
                "rationale",
            ],
            "additionalProperties": False,
        },
    },
}


AI_AUDIT_PROMPT = """You are a cybersecurity reviewer for Microsoft Fabric Spark notebooks. Analyze this notebook chunk and return structured JSON that estimates external resource access and data exfiltration risk for the chunk. Later processing will combine all chunk results for the notebook.

Primary goal: identify whether this chunk can read from or write to resources outside the current Fabric workspace, including other Fabric workspaces, other tenants, public internet services, external cloud storage, databases, APIs, messaging systems, email, or file transfer endpoints.

Score external_resource_access_score from 0 to 100:
- 0: only relative/default Lakehouse, Spark table, or current-workspace operations; no external indicators.
- 1-20: capability only, such as imports of network/cloud/database libraries or endpoint variables with no use.
- 21-45: possible external access, such as dynamic endpoints, connection strings, SDK clients, or configurable paths.
- 46-70: clear external access, such as literal URLs, cloud URIs, JDBC/ODBC targets, REST calls, cross-workspace/cross-tenant Fabric or Power BI APIs.
- 71-100: multiple external targets, high-confidence non-workspace targets, or known exfiltration-capable services.

Score exfiltration_risk_score from 0 to 100:
- 0: no external access and no suspicious capability.
- 1-20: benign-looking read-only access or unused capability.
- 21-45: unclear intent, dynamic endpoint construction, credentials involved, or possible external writes.
- 46-70: data read from internal/workspace sources and written/sent externally; broad credentials; obfuscated or hidden endpoints.
- 71-100: likely malicious or unauthorized movement, such as webhook POSTs, personal storage, paste sites, email/messaging APIs, encoded/compressed payloads sent externally, credential harvesting, or staged exfiltration.

Look for these signals:
- Protocols and endpoints: http, https, ftp, sftp, ssh, scp, smb, nfs, abfs, abfss, wasb, s3, gs, jdbc, odbc, mongodb, redis, kafka, eventhubs, smtp.
- External services: blob/dfs storage accounts, S3/GCS, Azure SQL/Snowflake/Redshift/Postgres/MySQL, Graph/Fabric/Power BI REST APIs, webhooks, Slack/Teams/Discord/Telegram, SendGrid/Mailgun, Dropbox/Box/Google Drive/OneDrive/SharePoint, GitHub gists, paste sites, serverless URLs.
- Code patterns: requests/httpx/urllib/aiohttp, boto3, google.cloud, azure.storage, paramiko, smtplib, socket, Spark read/write/load/save, writeStream, pandas/polars exports, notebookutils or mssparkutils credentials/secrets, TokenLibrary, os.environ, spark.conf credentials.
- Risk amplifiers: internal read followed by external write, POST/PUT/PATCH/upload/send/copy/mount, dynamic URL/path construction, secrets/tokens/connection strings, base64/gzip/pickle/encryption before network/file write, eval/exec/import indirection, shell commands such as curl/wget/scp/nc.

Decision rules:
- Treat Fabric, Power BI, Graph, Azure, and Microsoft URLs as potentially external unless the code proves they target the current workspace/tenant.
- Workspace-relative paths like Files/..., Tables/..., default saveAsTable, or spark.table without external location are not external by themselves.
- Literal external destinations should raise both scores. Literal external read-only sources may raise access more than risk.
- Dynamic endpoints should be represented as pseudocode and scored by capability; dynamic destinations are riskier than dynamic sources.
- If this chunk contains only one side of a suspicious flow, score that partial capability and explain the missing context.
- Do not infer benign ownership from domain names alone. Prefer a human-review flag when intent is unclear, but keep access score evidence-based.

For sources and destinations:
- Include deterministically known endpoints as literal strings.
- For dynamic endpoints, include pseudocode using angle placeholders, for example storage_account + '.dfs.core.windows.net/' + container + '/' + path or BASE_URL + '/api/v1/upload?token=' + TOKEN.
- Put read endpoints in sources and write/send/upload/copy/export endpoints in destinations. If direction is unclear, include the endpoint in both arrays and explain uncertainty in rationale.
- Return empty arrays when no endpoints are found.

Return only JSON matching the required schema. No markdown, no extra text.

Notebook file: {file_name}
Chunk: {chunk_number} of {chunk_count}
Notebook code chunk:
{notebook_definition}
"""

## Run AI Assessment

Apply the AI function to each notebook chunk. The chunk-level responses are structured with a JSON Schema and recombined in the next step.

In [ ]:
from synapse.ml.spark import aifunc

# Run AI assessment against each notebook chunk
audit_df = notebook_chunks_df.ai.generate_response(
    prompt=AI_AUDIT_PROMPT,
    is_prompt_template=True,
    output_col="ai_response",
    error_col="ai_error",
    response_format=AI_AUDIT_RESPONSE_FORMAT,
)

audit_df.cache()

n_success = audit_df.filter(F.col("ai_error").isNull()).count()
n_error = audit_df.filter(F.col("ai_error").isNotNull()).count()
print(f"AI chunk assessment complete: {n_success} chunk(s) succeeded, {n_error} failed.")

if n_error > 0:
    print("\nErrors:")
    (audit_df
        .filter(F.col("ai_error").isNotNull())
        .select("file_name", "chunk_number", "chunk_count", "ai_error")
        .show(truncate=False))

## Parse and Recombine Chunk Results

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, BooleanType, ArrayType
)

# Define schema matching the AI response JSON Schema for parsing
endpoint_schema = StructType([
    StructField("endpoint", StringType(), True),
    StructField("type", StringType(), True),
    StructField("deterministic", BooleanType(), True),
])

result_schema = StructType([
    StructField("external_resource_access_score", IntegerType(), True),
    StructField("exfiltration_risk_score", IntegerType(), True),
    StructField("sources", ArrayType(endpoint_schema), True),
    StructField("destinations", ArrayType(endpoint_schema), True),
    StructField("rationale", StringType(), True),
])

parsed_chunk_df = (
    audit_df
    .filter(F.col("ai_error").isNull())
    .withColumn("parsed", F.from_json(F.col("ai_response"), result_schema))
)

n_parse_error = parsed_chunk_df.filter(F.col("parsed").isNull()).count()
if n_parse_error > 0:
    print(f"Warning: {n_parse_error} successful AI response(s) could not be parsed as JSON.")

chunk_results_df = (
    parsed_chunk_df
    .filter(F.col("parsed").isNotNull())
    .select(
        F.col("path"),
        F.col("file_name"),
        F.col("content_length"),
        F.col("chunk_index"),
        F.col("chunk_number"),
        F.col("chunk_count"),
        F.col("parsed.external_resource_access_score").alias("external_resource_access_score"),
        F.col("parsed.exfiltration_risk_score").alias("exfiltration_risk_score"),
        F.col("parsed.sources").alias("sources_array"),
        F.col("parsed.destinations").alias("destinations_array"),
        F.col("parsed.rationale").alias("chunk_rationale"),
    )
)

empty_endpoint_array = F.from_json(F.lit("[]"), ArrayType(endpoint_schema))

results_df = (
    chunk_results_df
    .groupBy("path", "file_name", "content_length", "chunk_count")
    .agg(
        F.max("external_resource_access_score").alias("external_resource_access_score"),
        F.max("exfiltration_risk_score").alias("exfiltration_risk_score"),
        F.array_distinct(
            F.flatten(F.collect_list(F.coalesce(F.col("sources_array"), empty_endpoint_array)))
        ).alias("sources_array"),
        F.array_distinct(
            F.flatten(F.collect_list(F.coalesce(F.col("destinations_array"), empty_endpoint_array)))
        ).alias("destinations_array"),
        F.concat_ws(
            " | ",
            F.collect_list(
                F.concat(
                    F.lit("chunk "),
                    F.col("chunk_number").cast("string"),
                    F.lit(": "),
                    F.col("chunk_rationale")
                )
            )
        ).alias("chunk_rationales"),
    )
    .select(
        F.col("path"),
        F.col("file_name"),
        F.col("content_length"),
        F.col("chunk_count"),
        F.col("external_resource_access_score"),
        F.col("exfiltration_risk_score"),
        F.to_json(F.col("sources_array")).alias("sources"),
        F.to_json(F.col("destinations_array")).alias("destinations"),
        F.concat(
            F.lit("Aggregated from "),
            F.col("chunk_count").cast("string"),
            F.lit(" chunk(s). Highest chunk scores are shown. Chunk rationale: "),
            F.substring(F.col("chunk_rationales"), 1, 4000),
        ).alias("rationale"),
        F.current_timestamp().alias("assessed_at"),
    )
)

chunk_results_df.cache()
results_df.cache()

n_chunk_results = chunk_results_df.count()
n_notebook_results = results_df.count()
print(f"Parsed {n_chunk_results} chunk-level result(s).")
print(f"Recombined results for {n_notebook_results} notebook(s).")

## Write Results to Lakehouse Table

In [ ]:
results_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE)

print(f"Results written to table: {OUTPUT_TABLE}")
print(f"Total notebooks assessed: {results_df.count()}")

## Summary

In [ ]:
print("=" * 70)
print("AI AUDIT SUMMARY")
print("=" * 70)

# Score distribution
print("\nScore Statistics:")
results_df.select(
    "external_resource_access_score", "exfiltration_risk_score"
).summary("min", "mean", "max").show()

# High-risk notebooks (exfiltration_risk_score >= 50)
print("\nHigh-Risk Notebooks (exfiltration_risk_score >= 50):")
(results_df
    .filter(F.col("exfiltration_risk_score") >= 50)
    .select("file_name", "external_resource_access_score", "exfiltration_risk_score", "rationale")
    .orderBy(F.desc("exfiltration_risk_score"))
    .show(50, truncate=False))

# Notebooks with confirmed external access (score >= 50)
print("\nNotebooks with External Access (external_resource_access_score >= 50):")
(results_df
    .filter(F.col("external_resource_access_score") >= 50)
    .select("file_name", "external_resource_access_score", "exfiltration_risk_score", "rationale")
    .orderBy(F.desc("external_resource_access_score"))
    .show(50, truncate=False))

# Low-risk summary
n_low_risk = results_df.filter(
    (F.col("external_resource_access_score") < 25) & (F.col("exfiltration_risk_score") < 25)
).count()
print(f"\nLow-risk notebooks (both scores < 25): {n_low_risk} of {results_df.count()}")

## Explore Detailed Results (Optional)

Query the results table to inspect sources, destinations, and rationale for specific notebooks.

In [ ]:
# Example queries — uncomment the one you need:

# All results ordered by risk:
# spark.sql(f"SELECT file_name, external_resource_access_score, exfiltration_risk_score, rationale FROM {OUTPUT_TABLE} ORDER BY exfiltration_risk_score DESC").show(truncate=False)

# Inspect sources and destinations for a specific notebook:
# spark.sql(f"SELECT file_name, sources, destinations FROM {OUTPUT_TABLE} WHERE exfiltration_risk_score >= 50").show(truncate=False)

# Notebooks flagged for review (either score >= 40):
# spark.sql(f"SELECT * FROM {OUTPUT_TABLE} WHERE external_resource_access_score >= 40 OR exfiltration_risk_score >= 40").show(truncate=False)